In [ ]:
from datetime import datetime
import re
import os
import time
import requests
import fitz 
from urllib.parse import urlparse, parse_qs, urljoin
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import shutil

In [ ]:
# parse the dates

date_patterns = [
    r"\b\d{1,2} [a-zéûî]+ \d{4}\b",         
    r"\b\d{1,2}/\d{1,2}/\d{2,4}\b",          
    r"\b\d{1,2}-\d{1,2}-\d{2,4}\b",          
    r"\b\d{4}-\d{1,2}-\d{1,2}\b",            
    r"\ble \d{1,2} [a-zéûî]+ \d{4}\b",       
    r"\bavant le \d{1,2} [a-zéûî]+ \d{4}\b",
    r"\bau plus tard le \d{1,2} [a-zéûî]+ \d{4}\b", 
    r"\d{1,2} (janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre) \d{4}"
]

months_fr = {
    "janvier": 1, "février": 2, "mars": 3, "avril": 4, "mai": 5, "juin": 6,
    "juillet": 7, "août": 8, "septembre": 9, "octobre": 10, "novembre": 11, "décembre": 12
}

def parse_french_date(date_str):
    date_str = date_str.lower()

    if re.match(r"\d{1,2}/\d{1,2}/\d{4}", date_str):
        return datetime.strptime(date_str, "%d/%m/%Y")
    elif re.match(r"\d{1,2}-\d{1,2}-\d{4}", date_str):
        return datetime.strptime(date_str, "%d-%m-%Y")

    match = re.match(r"(\d{1,2}) (\w+) (\d{4})", date_str)
    if match:
        day, month_str, year = match.groups()
        month = months_fr.get(month_str, 0)
        if month:
            return datetime(int(year), month, int(day))
    return None 

In [ ]:
url = "LINK" #add link

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"
}

In [ ]:
# pagination

response = requests.get(url, headers=headers, timeout=10)
print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")
pagination = soup.find("div", class_="posts-page-links")

if pagination:

    page_numbers = []

    for el in pagination.find_all(["a", "span"], class_="page-numbers"):
        try:
            num = int(el.text.strip())
            page_numbers.append(num)
        except ValueError:
            continue 

    if page_numbers:
        total_pages = max(page_numbers)
        print("Total number of pages:", total_pages)
    else:
        print("No page found.")
else:
    print("Pagination not found.")

In [ ]:
# scraping

list_link = []

for page in range(1, total_pages + 1):
    if page == 1:
        n_url = url
    else:
        n_url = f"{url}page/{page}/"
    
    res = requests.get(n_url, headers=headers, timeout=10)
    page_soup = BeautifulSoup(res.text, "html.parser")
    
    entries = page_soup.find_all("div", class_="entry-holder")
    
    for entry in entries:
        title_tag = entry.find("h2", class_="entry-title")
        link = title_tag.find("a")["href"] if title_tag and title_tag.find("a") else ""
        list_link.append({
            "title": title_tag.get_text(strip=True) if title_tag else "",
            "publication_date": entry.find("li", class_="item-date").get_text(strip=True) if entry.find("li", class_="item-date") else "",
            "expiration_date": None, 
            "link": link
        })

print(list_link)

In [ ]:
driver = webdriver.Chrome()
driver.get(url)

In [ ]:
# scraping

pdf_folder = "saved_files"
if os.path.exists(pdf_folder):
    shutil.rmtree(pdf_folder)
os.makedirs("saved_files", exist_ok=True)

tenders = []
list_expiration_date =[]

for i, data in enumerate(list_link, 1):
    link = data["link"]
    title = data["title"]
    publish_date = data["publication_date"]

    driver.get(link)
    time.sleep(2)

    try:
        iframe = driver.find_element(By.TAG_NAME, "iframe")
        iframe_src = iframe.get_attribute("src")
    except:
        print("No iframe found")
        driver.quit()
        continue
    
    parsed = urlparse(iframe_src)
    pdf_param = parse_qs(parsed.query).get("file", [None])[0]

    if pdf_param:
        full_pdf_url = urljoin(link, pdf_param)
        filename = f"saved_files/document_{i}.pdf"
        r = requests.get(full_pdf_url)
        with open(filename, "wb") as f:
            f.write(r.content)
    
        with fitz.open(filename) as doc:
            text = ""
            for page in doc:
                text += page.get_text()
            text = text.replace("\n", " ").replace("’", "'").replace("É", "é")
    
        expiration_date = None
        found = False
        
        keywords = [
            "DEPOT ET VALIDITE DE L’OFFRE",
            "DEPOT ET VALIDITE DE L'OFFRE",
            "DÉPÔT ET VALIDITÉ DE L’OFFRE"
        ]
        
        text_lower = text.lower()
        
        for key in keywords:
            for match in re.finditer(re.escape(key.lower()), text_lower):
                start = match.start()
                section_text = text[start:start + 500]
        
                for pattern in date_patterns:
                    date_match = re.search(pattern, section_text, flags=re.IGNORECASE)
                    if date_match:
                        expiration_date = date_match.group().strip()
                        list_expiration_date.append(expiration_date)
                        found = True
                        break
        
                if found:
                    break  
        
            if found:
                break  

        if not found:
            continue
     
        if expiration_date:
            parsed_date = parse_french_date(expiration_date)
            if parsed_date and parsed_date.date() > datetime.now().date():
                
                tenders.append({
                    "title": title,
                    "publication_date": publish_date,
                    "expiration_date": expiration_date,
                    "link": link,
                    "country": "Guinea"
                })
        else:
            print("expiration_date is empty or not detected.")
    else:
        print("No PDF found.")

In [ ]:
driver.quit()

In [ ]:
df =pd.DataFrame(tenders)
df.to_excel("../2_scraped_files/Guinea.xlsx", index=False) # modify